# Discrete logarithm with native FlagQuantum

Both original scenarios, actual observations, classical recovery and independent full-state checks. No physical wire assignment or bitstring decoding is needed.


## Express modular interference directly

The editable modulo-five qfunc below initializes the function register to one,
prepares two coherent exponents, requests two modular exponentiations and applies
inverse QFTs. FlagQuantum builds the reversible multiplication and Fourier
circuits. Both original scenarios still run their complete sampling and modular
recovery below; the visible construction does not replace the modulo-thirteen
experiment with a smaller one.

`alpha` and `beta` are integer outputs in the modulo-five program. The separate
modulo-thirteen root declares fractional outputs and its readout preserves that
scale. Try valid invertible bases with the same modulus in the visible binding
and compare the full state with the reference. A changed generator need not
satisfy every assumption of the source recovery heuristic. The real logarithm
ratio used in illustrative plots is not the recovered modular logarithm.


In [ ]:
from typing import Annotated
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import (
    Output,
    QNum,
    adjoint,
    allocate,
    hadamard_transform,
    modular_exp,
    qft,
    qfunc,
)


@qfunc
def modulo5(
    element: int,
    generator: int,
    alpha: Output[Annotated[QNum, 2]],
    beta: Output[Annotated[QNum, 2]],
    func_res: Output[Annotated[QNum, 3]],
) -> None:
    allocate(alpha)
    allocate(beta)
    func_res |= 1
    hadamard_transform(alpha)
    hadamard_transform(beta)
    modular_exp(5, element, func_res, alpha)
    modular_exp(5, generator, func_res, beta)
    with adjoint():
        qft(alpha)
        qft(beta)


visible_program = synth.synthesize(
    modulo5,
    classical_inputs={"element": 2, "generator": 3},
    max_qubits=13,
    max_workspace=2,
    memory_limit_bytes=64 * 1024**2,
)
visible_result = fq.run(visible_program)
print(
    "Named exponent probabilities:",
    dict(visible_program.probabilities(visible_result, registers=("alpha", "beta"))),
)
from io import BytesIO

from IPython.display import Image, display

from examples.synthesis.algorithms.number_theory_and_cryptography.discrete_log.overview import (
    workflow as dl,
)


## Modulo five

The original two-bit integer phase registers, target initialized to one and 4000 shots.


In [ ]:
mod5 = dl.run_estimate(5, seed=1605)
print("Shots:", mod5.shots, "Logarithm:", mod5.logarithm)
print(mod5.resources)
print(
    "Complete-state error:",
    mod5.maximum_state_error,
    "Dirty work:",
    mod5.dirty_workspace_probability,
)
for row in mod5.candidates:
    print(row)
assert mod5.logarithm == 3
assert pow(3, mod5.logarithm, 5) == 2


## Modulo thirteen

The original five-bit fractional phase registers, four-bit target and 10000 shots. All retained rows remain available, including incorrect low-count approximation tails. The original assertion covers the leading ten rows.


In [ ]:
mod13 = dl.run_estimate(13, seed=1613)
print("Shots:", mod13.shots, "Logarithm:", mod13.logarithm)
print(mod13.resources)
print(
    "Complete-state error:",
    mod13.maximum_state_error,
    "Dirty work:",
    mod13.dirty_workspace_probability,
)
for row in mod13.candidates[:10]:
    print(row)
print("Retained rows:", len(mod13.candidates))
print("Incorrect retained tails:", sum(row.logarithm != 8 for row in mod13.candidates))
assert all(pow(7, row.logarithm, 13) == 3 for row in mod13.candidates[:10])


## Source analytic illustrations

These curves use the real ratio log(3)/log(7), not the recovered modular logarithm 8. They are not measured marginals. The source sizes t=5 and t=8 are preserved.


In [ ]:
for bits in (5, 8):
    curves = dl.illustrative_curves(bits)
    figure = dl.plot_curves(curves)
    with BytesIO() as buffer:
        figure.savefig(buffer, format="png")
        display(Image(data=buffer.getvalue()))
    figure.clear()
    print("t:", bits, "Real log ratio:", curves.real_log_ratio)

# Independent complete-state check for the editable modulo-five program.
import torch

visible_state = visible_result.statevector().reshape(128, -1)
expected_state = fq.run(dl.reference_circuit(5)).statevector().flatten()
torch.testing.assert_close(visible_state[:, 0], expected_state, atol=2e-11, rtol=0)
assert float(visible_state[:, 1:].abs().square().sum()) < 1e-24
print("Editable qfunc complete-state and workspace comparison: passed")


## Reproducibility and scope

Local seeds replace unseeded hosted sampling. Independent Circuit and analytic FFT references check absolute complex amplitudes, clean workspace and identical local sample sequences. Source pins and all seventeen original code cells are documented in `provenance.json` and `WORKFLOW.md`.
